# Random Forest Regression Implementation

In [131]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings


warnings.filterwarnings('ignore')

%matplotlib inline

In [132]:
df = pd.read_csv(r"C:\Users\User\Desktop\machine_learning\Data\cardekho_dataset.csv",index_col=[0])

In [133]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [134]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [135]:
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)

In [136]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [137]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [138]:
# Getting all different types of features

num_features = [feature for feature in  df.columns if df[feature].dtype != 'O']
print('Num of Numerical Feature : ',len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Feature : ',len(cat_features))

discrete_features = [feature for feature in num_features if len(df[feature].unique()) <= 25]
print('Num of Discrete features : ',len(discrete_features))

continuos_features = [feature for feature in num_features if feature not in discrete_features]
print('Num of Continios feature : ', len(continuos_features))

Num of Numerical Feature :  7
Num of Categorical Feature :  4
Num of Discrete features :  2
Num of Continios feature :  5


In [139]:
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'],axis=1)
y = df['selling_price']

In [140]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


#Feature Encoding and Scaling

In [141]:
len(df['model'].unique())

120

In [142]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [143]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [144]:
## Create column Transformer with 3 types of transformer

num_features = X.select_dtypes(exclude='object').columns
onehot_columns = ['seller_type','fuel_type','transmission_type']
label_encoder_columns = ['model']

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer,onehot_columns),
        ('StandardScaler',numeric_transformer,num_features)
    ], remainder = 'passthrough'
)

In [145]:
X = preprocessor.fit_transform(X)

In [146]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=40)
X_train.shape,X_test.shape

((12328, 14), (3083, 14))

In [147]:
X_train

array([[ 0.        ,  0.        ,  0.        , ..., -1.32425883,
        -1.24008119, -0.40302241],
       [ 0.        ,  0.        ,  0.        , ...,  0.02099878,
         0.40519209, -0.40302241],
       [ 1.        ,  0.        ,  1.        , ...,  0.02291783,
        -0.04626904, -0.40302241],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.20138996,
         0.48198702, -0.40302241],
       [ 0.        ,  0.        ,  0.        , ..., -0.55087963,
        -0.61874036, -0.40302241],
       [ 0.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43582879, -0.40302241]], shape=(12328, 14))

# Model Training and model Selection

In [148]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [149]:
## Create a function to Evaluate Model
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mean_squared_error(true,predicted))
    r2_square = r2_score(true,predicted)
    return mae,rmse,r2_square

In [150]:
## Beginning the model Training 

models = {
    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(),
    "Ridge" : Ridge(),
    "K-Neighbours Regressor" : KNeighborsRegressor(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest Regressor " : RandomForestRegressor()
}

for i in range(len(list(models))) :
    model = list(models.values())[i]
    model.fit(X_train,y_train) #Train Model

    #Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #Evaluate the model
    model_train_mae,model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    model_test_mae,model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)

   



    print(list(models.keys())[i])

    print('---------------------------------------')
    print('Model Prediction for Training set')
    print('Rmse: {:.4f}'.format(model_train_rmse))
    print('MAE : {:.4f}'.format(model_train_mae))
    print('R2 Score : {:.4f}'.format(model_train_r2))
   
    

   

    print('---------------------------------------')
    print('Model Prediction for Test set')
    print('Rmse : {:.4f}'.format(model_test_rmse))
    print('Mae: {:.4f}'.format(model_test_mae))
    print('R2 score : {:.4f}'.format(model_test_r2))
   
    print('---------------------------------------')
    


Linear Regression
---------------------------------------
Model Prediction for Training set
Rmse: 563383.4410
MAE : 274314.7271
R2 Score : 0.6230
---------------------------------------
Model Prediction for Test set
Rmse : 456216.3540
Mae: 259285.1738
R2 score : 0.6695
---------------------------------------
Lasso
---------------------------------------
Model Prediction for Training set
Rmse: 563383.4454
MAE : 274313.2275
R2 Score : 0.6230
---------------------------------------
Model Prediction for Test set
Rmse : 456216.7873
Mae: 259285.0433
R2 score : 0.6695
---------------------------------------
Ridge
---------------------------------------
Model Prediction for Training set
Rmse: 563383.8313
MAE : 274277.2770
R2 Score : 0.6230
---------------------------------------
Model Prediction for Test set
Rmse : 456209.2551
Mae: 259270.5847
R2 score : 0.6695
---------------------------------------
K-Neighbours Regressor
---------------------------------------
Model Prediction for Training s

In [151]:
knn_params = {"n_neighbors" : [2,3,10,20,40,50]}
rf_params = {"max_depth" : [5,8,15,None,10],
             "max_features" : [5,7,"auto",8],
             "min_samples_split" : [2,8,15,20],
             "n_estimators" : [100,200,500,1000]}


In [152]:
randomcv_models = [('KNN', KNeighborsRegressor(),knn_params),
                   ("RF",RandomForestRegressor(),rf_params)]

In [153]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
## hyper parameter Tuning

model_param = {}
for name,model,params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=params,
                                n_iter = 100,
                                cv=3,
                                verbose=2,
                                n_jobs=1)
    
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

    for model_name in model_param:
        print(f"---------------Best Params{model_name}---------")
        print(model_param[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ......................................n_neighbors=2; total time=   0.2s
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END ......................................n_neighbors=3; total time=   0.2s
[CV] END ......................................n_neighbors=3; total time=   0.3s
[CV] END ......................................n_neighbors=3; total time=   0.2s
[CV] END .....................................n_neighbors=10; total time=   0.4s
[CV] END .....................................n_neighbors=10; total time=   0.5s
[CV] END .....................................n_neighbors=10; total time=   0.3s
[CV] END .....................................n_neighbors=20; total time=   0.5s
[CV] END .....................................n_neighbors=20; total time=   0.6s
[CV] END .....................................n_n